# Embedding Space PCA Visualization

**AI-2 Session 3.2 -- Instructor Demo**

---

> **Pre-run this notebook before class.** The embedding step takes ~30 seconds depending on corpus size and API latency. All cells should have output cached so the instructor can scroll through without waiting.

---

This notebook visualizes where documents and questions live in embedding space. We take every chunk from the Northbrook corpus, embed them, reduce to 3 dimensions with PCA, and plot them in an interactive 3D scatter.

Then we add **questions** to the same space.

The gap you see is why naive similarity search sometimes fails. Questions and answers live in different neighborhoods of embedding space. This is the fundamental limitation that motivates filtered retrieval (today) and HyDE / question enrichment (AI-3).

### Connection to Session 2.1

In `how_models_think.ipynb`, you saw that embedding models and LLMs process text fundamentally differently. The embedding model **compresses** text into a fixed-length vector -- GPS coordinates for meaning. It keeps topic and vocabulary but loses word order, negation, and relationships.

Now you're going to see what that coordinate space actually looks like. We project 203 corpus chunks into 3D and add questions. The clusters and gaps you'll see are a direct consequence of that compression.

Remember the word swap test? *"The manager approved the employee's request"* and *"The employee approved the manager's request"* scored 0.98 similarity. That same lossy compression creates the gap you're about to see -- questions and answers use different structures, but the embedding model sees mostly the same words.

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv()

# Core imports
import numpy as np
import pickle
from sklearn.decomposition import PCA

# Project imports
from src.s2_embeddings.embed import get_embedding, embed_texts, cosine_similarity
from src.s3_ingestion.chunker import chunk_document

# Visualization
import plotly.graph_objects as go
print("Using Plotly for interactive 3D visualization.")

print("All imports loaded.")

---

## 2. Load and Chunk the Corpus

We load every Northbrook document, chunk it using the same paragraph-aware strategy from Session 2.2, and tag each chunk with its `doc_type` and `source` metadata.

In [ ]:
def classify_doc_type(filename: str) -> str:
    """Same classification logic used during ingestion in Session 2.2."""
    name = filename.lower()
    if "memo" in name:
        return "memo"
    elif "policy" in name or "handbook" in name:
        return "policy"
    elif "financial" in name:
        return "financial"
    elif "meeting" in name or "standup" in name or "committee" in name:
        return "meeting"
    else:
        return "unknown"


data_dir = _root / "data" / "northbrook"
all_files = sorted(data_dir.glob("*.md"))

all_chunks = []       # list of Chunk objects
chunk_texts = []      # parallel list of text strings for embedding
chunk_doc_types = []  # parallel list of doc_type labels
chunk_sources = []    # parallel list of source filenames

for doc_path in all_files:
    text = doc_path.read_text()
    doc_type = classify_doc_type(doc_path.name)

    chunks = chunk_document(
        text,
        source=doc_path.name,
        doc_type=doc_type,
        chunk_size=500,
        overlap=100,
    )

    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_texts.append(chunk.text)
        chunk_doc_types.append(doc_type)
        chunk_sources.append(doc_path.name)

print(f"Loaded {len(all_files)} documents, produced {len(all_chunks)} chunks.")
print()

# Summary by doc_type
from collections import Counter
type_counts = Counter(chunk_doc_types)
for doc_type, count in sorted(type_counts.items()):
    print(f"  {doc_type:<12s}  {count} chunks")

---

## 3. Embed All Chunks

This is the expensive step. We embed every chunk using the same Voyage AI model the pipeline uses (`voyage-3-lite`). On a typical Northbrook corpus this takes 15-30 seconds.

> **This cell should be pre-run before class.** The instructor should not need to wait for embeddings during the demo.

In [ ]:
import time

cache_path = _root / "cache" / "embedding_space_viz.npz"
cache_path.parent.mkdir(exist_ok=True)

if cache_path.exists():
    cached = np.load(cache_path, allow_pickle=True)
    if len(cached["embeddings"]) == len(chunk_texts):
        chunk_embeddings = cached["embeddings"].tolist()
        print(f"Loaded {len(chunk_embeddings)} embeddings from cache.")
    else:
        print(f"Cache has {len(cached['embeddings'])} embeddings but corpus has {len(chunk_texts)} chunks. Recomputing...")
        print(f"Embedding {len(chunk_texts)} chunks with voyage-3-lite...")
        t0 = time.time()
        chunk_embeddings = embed_texts(chunk_texts)
        elapsed = time.time() - t0
        np.savez(cache_path, embeddings=np.array(chunk_embeddings))
        print(f"Done in {elapsed:.1f}s. Saved to cache.")
else:
    print(f"Embedding {len(chunk_texts)} chunks with voyage-3-lite...")
    t0 = time.time()
    chunk_embeddings = embed_texts(chunk_texts)
    elapsed = time.time() - t0
    np.savez(cache_path, embeddings=np.array(chunk_embeddings))
    print(f"Done in {elapsed:.1f}s. Saved to cache.")

chunk_matrix = np.array(chunk_embeddings)

print(f"Embedding matrix shape: {chunk_matrix.shape}  (chunks x dimensions)")

---

## 4. PCA to 3D

We reduce the 512-dimensional embeddings to 3 dimensions using PCA. This loses detail, but preserves enough structure to see clusters and gaps.

We save the fitted PCA transformer so:
1. We can project new questions into the same space without recomputing
2. The walk-along notebook (`session_3_2.ipynb`) can load it for the student exploration cell

In [ ]:
pca = PCA(n_components=3)
coords_3d = pca.fit_transform(chunk_matrix)

explained = pca.explained_variance_ratio_
total_explained = sum(explained)

print(f"PCA explained variance:")
print(f"  PC1: {explained[0]:.1%}")
print(f"  PC2: {explained[1]:.1%}")
print(f"  PC3: {explained[2]:.1%}")
print(f"  Total: {total_explained:.1%}")
print()
print(f"(This is how much structure the 3D projection captures from the original {chunk_matrix.shape[1]} dimensions.)")
print()
print("Think of this like a photograph of a 3D sculpture. This camera angle captures")
print(f"about {total_explained:.0%} of the sculpture's distinguishing features. You can tell")
print("the clusters apart and see the gap, but the exact distances are approximate --")
print(f"{1 - total_explained:.0%} of the detail is lost in this projection. The patterns you see")
print("are real. The precise positions are rough.")

---

## 5. 3D Scatter Plot -- Corpus Chunks by Document Type

Every dot is a chunk from the Northbrook corpus. Colors represent document types:
- **Blue** = policy documents (handbook, vacation policy, expense policy, remote work policy)
- **Green** = meeting notes (board meetings, standup, HR committee)
- **Orange** = financial reports (Q3 and Q4 2024)
- **Red** = memos (CEO kickoff, office relocation, security update, policy correction)

Look for clusters. Similar content should group together.

In [ ]:
# Color mapping for doc_types
DOC_TYPE_COLORS = {
    "policy":    "#2196F3",  # blue
    "meeting":   "#4CAF50",  # green
    "financial": "#FF9800",  # orange
    "memo":      "#E91E63",  # red/pink
    "unknown":   "#9E9E9E",  # gray
}

DOC_TYPE_NAMES = {
    "policy":    "Policy",
    "meeting":   "Meeting Notes",
    "financial": "Financial Reports",
    "memo":      "Memos",
    "unknown":   "Other",
}


def build_corpus_figure():
    """Build the base 3D scatter plot with corpus chunks colored by doc_type."""

    fig = go.Figure()

    for doc_type in DOC_TYPE_COLORS:
        mask = [dt == doc_type for dt in chunk_doc_types]
        if not any(mask):
            continue

        indices = [i for i, m in enumerate(mask) if m]
        xs = coords_3d[indices, 0]
        ys = coords_3d[indices, 1]
        zs = coords_3d[indices, 2]

        # Build hover text: source filename + first 80 chars of chunk
        hover = [
            f"{chunk_sources[i]}<br>{chunk_texts[i][:80]}..."
            for i in indices
        ]

        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="markers",
            marker=dict(
                size=5,
                color=DOC_TYPE_COLORS[doc_type],
                opacity=0.7,
                line=dict(width=0.5, color="white"),
            ),
            name=DOC_TYPE_NAMES[doc_type],
            text=hover,
            hoverinfo="text+name",
        ))

    fig.update_layout(
        title="Northbrook Corpus -- Embedding Space (PCA to 3D)",
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
        ),
        width=900,
        height=700,
        legend=dict(x=0.01, y=0.99),
    )
    return fig


fig_corpus = build_corpus_figure()
fig_corpus.show()

print("\nRotate the plot to explore the clusters.")
print("Hover over points to see the source document and chunk preview.")

---

## 6. Add Questions to the Plot

Now the key demo. We define test questions -- the kind of queries someone would actually ask the Northbrook corpus -- embed them, and project them into the same PCA space.

**Watch where the questions land relative to the chunks that contain their answers.**

In [ ]:
# Test questions -- designed to target specific document types
test_questions = [
    {
        "question": "What is Northbrook's vacation policy?",
        "expected_type": "policy",
        "notes": "Answer is in vacation_policy_2025.md -- a policy document",
    },
    {
        "question": "How many vacation days do employees get per year?",
        "expected_type": "policy",
        "notes": "Specific factual answer: 20 days. Lives in policy.",
    },
    {
        "question": "What was the Q3 2024 revenue?",
        "expected_type": "financial",
        "notes": "Answer is $12.4M -- in financial_report_q3_2024.md",
    },
    {
        "question": "What did the CEO say about company direction for 2025?",
        "expected_type": "memo",
        "notes": "Answer is in memo_ceo_annual_kickoff.md",
    },
    {
        "question": "What security changes are required for all employees?",
        "expected_type": "memo",
        "notes": "Answer is in memo_security_update.md",
    },
    {
        "question": "When is the office relocation happening?",
        "expected_type": "memo",
        "notes": "Answer is in memo_office_relocation.md",
    },
    {
        "question": "What was discussed at the Q4 board meeting?",
        "expected_type": "meeting",
        "notes": "Answer is in board_meeting_q4_2024.md",
    },
    {
        "question": "How is the cloud migration project progressing?",
        "expected_type": "meeting",
        "notes": "Answer is in engineering_standup_2025_01.md",
    },
    {
        "question": "What are the expense reimbursement limits for travel?",
        "expected_type": "policy",
        "notes": "Answer is in expense_policy.md",
    },
    {
        "question": "What changes were made to the remote work schedule?",
        "expected_type": "policy",
        "notes": "Answer is in remote_work_policy.md",
    },
]

print(f"Defined {len(test_questions)} test questions:\n")
for i, q in enumerate(test_questions, 1):
    print(f"  {i:>2}. [{q['expected_type']:<10s}] {q['question']}")

In [ ]:
# Embed all test questions
question_texts = [q["question"] for q in test_questions]

print(f"Embedding {len(question_texts)} questions...")
question_embeddings = embed_texts(question_texts)
question_matrix = np.array(question_embeddings)

# Project into the same PCA space as the corpus
question_3d = pca.transform(question_matrix)

print(f"Questions projected into PCA space.")
print(f"Question coordinates shape: {question_3d.shape}")

In [ ]:
def build_full_figure():
    """Build the 3D scatter with both corpus chunks and questions."""

    fig = go.Figure()

    # --- Corpus chunks (same as before, slightly smaller) ---
    for doc_type in DOC_TYPE_COLORS:
        mask = [dt == doc_type for dt in chunk_doc_types]
        if not any(mask):
            continue

        indices = [i for i, m in enumerate(mask) if m]
        xs = coords_3d[indices, 0]
        ys = coords_3d[indices, 1]
        zs = coords_3d[indices, 2]

        hover = [
            f"{chunk_sources[i]}<br>{chunk_texts[i][:80]}..."
            for i in indices
        ]

        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="markers",
            marker=dict(
                size=4,
                color=DOC_TYPE_COLORS[doc_type],
                opacity=0.5,
                line=dict(width=0.5, color="white"),
            ),
            name=f"Chunks: {DOC_TYPE_NAMES[doc_type]}",
            text=hover,
            hoverinfo="text+name",
        ))

    # --- Questions (distinct markers) ---
    q_hover = [
        f"Q: {q['question']}<br>Expected: {q['expected_type']}<br>{q['notes']}"
        for q in test_questions
    ]

    fig.add_trace(go.Scatter3d(
        x=question_3d[:, 0],
        y=question_3d[:, 1],
        z=question_3d[:, 2],
        mode="markers+text",
        marker=dict(
            size=10,
            color="#FFD700",
            symbol="diamond",
            opacity=1.0,
            line=dict(width=2, color="black"),
        ),
        name="QUESTIONS",
        text=[f"Q{i+1}" for i in range(len(test_questions))],
        textposition="top center",
        textfont=dict(size=10, color="black"),
        hovertext=q_hover,
        hoverinfo="text",
    ))

    fig.update_layout(
        title="Embedding Space: Corpus Chunks + Questions (PCA to 3D)",
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
        ),
        width=950,
        height=750,
        legend=dict(x=0.01, y=0.99),
    )
    return fig


# --- Compute gap severity for each question to find worst cases ---
gap_cases = []
for i, q in enumerate(test_questions):
    q_vec = question_embeddings[i]
    expected_type = q["expected_type"]

    # Find nearest chunk of CORRECT type
    correct_sims = []
    correct_indices = []
    for j in range(len(chunk_embeddings)):
        if chunk_doc_types[j] == expected_type:
            sim = cosine_similarity(q_vec, chunk_embeddings[j])
            correct_sims.append(sim)
            correct_indices.append(j)

    # Find nearest chunk of WRONG type
    wrong_sims = []
    wrong_indices = []
    for j in range(len(chunk_embeddings)):
        if chunk_doc_types[j] != expected_type:
            sim = cosine_similarity(q_vec, chunk_embeddings[j])
            wrong_sims.append(sim)
            wrong_indices.append(j)

    best_correct_idx = correct_indices[int(np.argmax(correct_sims))]
    best_correct_sim = max(correct_sims)
    best_wrong_idx = wrong_indices[int(np.argmax(wrong_sims))]
    best_wrong_sim = max(wrong_sims)

    gap = best_correct_sim - best_wrong_sim  # negative = wrong chunk is closer
    gap_cases.append({
        "q_idx": i,
        "correct_chunk_idx": best_correct_idx,
        "correct_sim": best_correct_sim,
        "wrong_chunk_idx": best_wrong_idx,
        "wrong_sim": best_wrong_sim,
        "gap": gap,
    })

# Sort by gap (worst first -- smallest or most negative gap)
gap_cases_sorted = sorted(gap_cases, key=lambda x: x["gap"])
worst_cases = gap_cases_sorted[:4]

# Build figure and add lines for worst cases
fig_full = build_full_figure()

for case in worst_cases:
    qi = case["q_idx"]
    q_label = f"Q{qi+1}"

    # Green dashed line to nearest CORRECT chunk
    ci = case["correct_chunk_idx"]
    fig_full.add_trace(go.Scatter3d(
        x=[question_3d[qi, 0], coords_3d[ci, 0]],
        y=[question_3d[qi, 1], coords_3d[ci, 1]],
        z=[question_3d[qi, 2], coords_3d[ci, 2]],
        mode="lines",
        line=dict(color="#00CC00", width=3, dash="dash"),
        name=f"{q_label} -> correct ({case['correct_sim']:.3f})",
        hoverinfo="skip",
        showlegend=True,
    ))

    # Red dashed line to nearest WRONG chunk
    wi = case["wrong_chunk_idx"]
    fig_full.add_trace(go.Scatter3d(
        x=[question_3d[qi, 0], coords_3d[wi, 0]],
        y=[question_3d[qi, 1], coords_3d[wi, 1]],
        z=[question_3d[qi, 2], coords_3d[wi, 2]],
        mode="lines",
        line=dict(color="#CC0000", width=3, dash="dash"),
        name=f"{q_label} -> wrong ({case['wrong_sim']:.3f})",
        hoverinfo="skip",
        showlegend=True,
    ))

fig_full.show()

print("\nGold diamonds = questions. Colored dots = corpus chunks.")
print("Hover over any question marker to see the question text and expected source type.")
print()
print("Lines show the 4 worst gap cases:")
print("  Green dashed = nearest chunk of the CORRECT doc_type")
print("  Red dashed   = nearest chunk of a WRONG doc_type")
print("  When the red line is shorter than green, naive retrieval gets the wrong answer.")

---

## 7. The Gap

Look at the plot above.

**The question markers (gold diamonds) cluster together, away from the corpus chunks.** Questions live in "question space." Answers live in "answer space." They are neighbors -- close enough for semantic search to usually work -- but not the same neighborhood.

This matters because:
- A question about vacation policy is semantically closer to OTHER questions about vacation than it is to the statement "Employees accrue 20 days of PTO per year."
- A question about Q3 revenue is closer to other financial questions than to the sentence "Revenue grew 8% year-over-year."

Let's measure the gap directly.

In [ ]:
print("Question-to-Corpus Distance Analysis")
print("=" * 80)
print()

gap_results = []

for i, q in enumerate(test_questions):
    q_vec = question_embeddings[i]
    expected_type = q["expected_type"]

    # Similarity to chunks of the EXPECTED doc_type (where the answer lives)
    expected_sims = [
        cosine_similarity(q_vec, chunk_embeddings[j])
        for j in range(len(chunk_embeddings))
        if chunk_doc_types[j] == expected_type
    ]

    # Similarity to chunks of OTHER doc_types (where the answer does NOT live)
    other_sims = [
        cosine_similarity(q_vec, chunk_embeddings[j])
        for j in range(len(chunk_embeddings))
        if chunk_doc_types[j] != expected_type
    ]

    # Similarity to OTHER questions
    other_q_sims = [
        cosine_similarity(q_vec, question_embeddings[j])
        for j in range(len(question_embeddings))
        if j != i
    ]

    best_expected = max(expected_sims) if expected_sims else 0
    best_other = max(other_sims) if other_sims else 0
    best_other_q = max(other_q_sims) if other_q_sims else 0

    # Flag cases where a WRONG chunk or another QUESTION is closer than the RIGHT chunk
    wrong_chunk_closer = best_other > best_expected
    question_closer = best_other_q > best_expected
    flag_wrong_chunk = " <-- WRONG chunk closer!" if wrong_chunk_closer else ""
    flag_question = " <-- another QUESTION closer!" if question_closer else ""

    gap_results.append({
        "q_idx": i,
        "question": q["question"],
        "expected_type": expected_type,
        "best_expected": best_expected,
        "best_other": best_other,
        "best_other_q": best_other_q,
        "wrong_chunk_closer": wrong_chunk_closer,
        "question_closer": question_closer,
        "gap": best_expected - best_other,
        "q_gap": best_expected - best_other_q,
    })

    print(f"Q{i+1}: {q['question']}")
    print(f"     Expected type: {expected_type}")
    print(f"     Best match in {expected_type} chunks: {best_expected:.4f}")
    print(f"     Best match in OTHER chunks:         {best_other:.4f}{flag_wrong_chunk}")
    print(f"     Best match among OTHER questions:   {best_other_q:.4f}{flag_question}")
    print()

# --- Summary Table (sorted by gap severity, worst first) ---
print()
print("=" * 80)
print("SUMMARY: Gap Severity (sorted worst-first)")
print("=" * 80)
print()
print(f"{'Q#':<4s} {'Expected':<11s} {'Best Correct':<14s} {'Best Wrong':<12s} {'Best OtherQ':<13s} {'Gap':<8s} {'Status'}")
print("-" * 80)

gap_sorted = sorted(gap_results, key=lambda x: x["gap"])
for r in gap_sorted:
    status = ""
    if r["wrong_chunk_closer"]:
        status = "WRONG CLOSER"
    elif r["question_closer"]:
        status = "Q CLOSER"
    else:
        status = "ok"
    print(f"Q{r['q_idx']+1:<3d} {r['expected_type']:<11s} {r['best_expected']:<14.4f} {r['best_other']:<12.4f} {r['best_other_q']:<13.4f} {r['gap']:>+7.4f}  {status}")

# --- Key Statistics ---
wrong_chunk_count = sum(1 for r in gap_results if r["wrong_chunk_closer"])
question_closer_count = sum(1 for r in gap_results if r["question_closer"])
total_q = len(gap_results)

print()
print("-" * 80)
print(f"  {wrong_chunk_count} out of {total_q} questions have a WRONG chunk type closer than the correct type")
print(f"  {question_closer_count} out of {total_q} questions are closer to ANOTHER QUESTION than to their best answer chunk")
print()
print("This is the embedding gap. Questions cluster with questions, not with answers.")

### What to look for in the output above

1. **Cases where the wrong chunk type scores higher** -- the question is closer to an irrelevant document than to the document containing the answer. This is where naive retrieval retrieves the wrong thing.

2. **Cases where another question is closer than any answer chunk** -- questions are more similar to each other than to the answers they seek. This is the embedding gap.

3. **Cases where the right type wins clearly** -- filtering would help here because the correct doc_type has the highest similarity.

Neither filtering nor naive retrieval solves all cases. Filtering narrows the pool (which helps when the right type is known), but the gap between question-phrasing and answer-phrasing remains within any pool. Addressing that gap requires techniques introduced in AI-3: **HyDE** (transforming the query to sound like a hypothetical answer) and **question enrichment** (adding "what questions does this chunk answer?" to chunks at ingestion time).

---

## 7b. HyDE Preview: What If the Question Sounded Like an Answer?

What if we rephrased the question to *sound like* an answer before searching? Watch where the green star lands compared to the gold diamond.

In [ ]:
# HyDE Preview: embed questions AND hypothetical answers, compare positions

hyde_examples = [
    {
        "question": "What is Northbrook's vacation policy?",
        "hypothetical": "Northbrook's vacation policy provides employees with 20 days of paid time off per year, with additional days accruing based on tenure.",
        "expected_type": "policy",
    },
    {
        "question": "What was the Q3 2024 revenue?",
        "hypothetical": "Northbrook's Q3 2024 revenue was $12.4 million, representing an 8% year-over-year increase driven by new client acquisitions.",
        "expected_type": "financial",
    },
]

# Embed questions and hypothetical answers
hyde_q_texts = [h["question"] for h in hyde_examples]
hyde_h_texts = [h["hypothetical"] for h in hyde_examples]

hyde_q_vecs = embed_texts(hyde_q_texts)
hyde_h_vecs = embed_texts(hyde_h_texts)

hyde_q_3d = pca.transform(np.array(hyde_q_vecs))
hyde_h_3d = pca.transform(np.array(hyde_h_vecs))

# Build figure: faded corpus + question diamonds + hypothetical answer stars
fig_hyde = go.Figure()

# Corpus chunks (faded)
for doc_type in DOC_TYPE_COLORS:
    mask = [dt == doc_type for dt in chunk_doc_types]
    if not any(mask):
        continue
    indices = [i for i, m in enumerate(mask) if m]
    fig_hyde.add_trace(go.Scatter3d(
        x=coords_3d[indices, 0], y=coords_3d[indices, 1], z=coords_3d[indices, 2],
        mode="markers",
        marker=dict(size=4, color=DOC_TYPE_COLORS[doc_type], opacity=0.3),
        name=f"Chunks: {DOC_TYPE_NAMES[doc_type]}",
        hoverinfo="skip",
    ))

# Questions (gold diamonds)
for k, h in enumerate(hyde_examples):
    q_label = f"Q: {h['question'][:35]}..."
    fig_hyde.add_trace(go.Scatter3d(
        x=[hyde_q_3d[k, 0]], y=[hyde_q_3d[k, 1]], z=[hyde_q_3d[k, 2]],
        mode="markers+text",
        marker=dict(size=12, color="#FFD700", symbol="diamond", opacity=1.0, line=dict(width=2, color="black")),
        name=q_label,
        text=[f"Q{k+1}"],
        textposition="top center",
        textfont=dict(size=11, color="#B8860B"),
        hovertext=[h["question"]],
        hoverinfo="text",
        showlegend=True,
    ))

    # Hypothetical answer (green star)
    h_label = f"HyDE: {h['hypothetical'][:35]}..."
    fig_hyde.add_trace(go.Scatter3d(
        x=[hyde_h_3d[k, 0]], y=[hyde_h_3d[k, 1]], z=[hyde_h_3d[k, 2]],
        mode="markers+text",
        marker=dict(size=12, color="#00CC66", symbol="diamond", opacity=1.0, line=dict(width=2, color="black")),
        name=h_label,
        text=[f"H{k+1}"],
        textposition="top center",
        textfont=dict(size=11, color="#006633"),
        hovertext=[h["hypothetical"]],
        hoverinfo="text",
        showlegend=True,
    ))

    # Arrow-like line from question to hypothetical
    fig_hyde.add_trace(go.Scatter3d(
        x=[hyde_q_3d[k, 0], hyde_h_3d[k, 0]],
        y=[hyde_q_3d[k, 1], hyde_h_3d[k, 1]],
        z=[hyde_q_3d[k, 2], hyde_h_3d[k, 2]],
        mode="lines",
        line=dict(color="#00CC66", width=3, dash="dash"),
        showlegend=False, hoverinfo="skip",
    ))

fig_hyde.update_layout(
    title="HyDE Preview: Questions (gold) vs Hypothetical Answers (green)",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=950, height=750,
    legend=dict(x=0.01, y=0.99),
)
fig_hyde.show()

# Print similarity comparison
print("\nSimilarity comparison: question vs hypothetical answer")
print("=" * 70)
for k, h in enumerate(hyde_examples):
    q_vec = hyde_q_vecs[k]
    h_vec = hyde_h_vecs[k]
    expected_type = h["expected_type"]

    # Best match for question
    q_sims = [cosine_similarity(q_vec, chunk_embeddings[j]) for j in range(len(chunk_embeddings)) if chunk_doc_types[j] == expected_type]
    # Best match for hypothetical
    h_sims = [cosine_similarity(h_vec, chunk_embeddings[j]) for j in range(len(chunk_embeddings)) if chunk_doc_types[j] == expected_type]

    print(f"\n  Q{k+1}: {h['question']}")
    print(f"  H{k+1}: {h['hypothetical'][:70]}...")
    print(f"  Question best match in {expected_type}: {max(q_sims):.4f}")
    print(f"  HyDE best match in {expected_type}:     {max(h_sims):.4f}  ({max(h_sims) - max(q_sims):+.4f})")

The green star lands closer to the answer chunks. That technique is called **HyDE** (Hypothetical Document Embedding). You will build it in AI-3.

---

## 8. Interactive: Add Your Own Question

Type any question below. The cell embeds it, projects it into the same PCA space, and adds it to the plot as a bright red star. Try questions that:
- Should clearly target one document type
- Are ambiguous or cross-document
- Use very different vocabulary than the source documents

Where does your question land?

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_input = widgets.Text(
    value="",
    placeholder="Type a question about the Northbrook corpus...",
    description="Question:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "80px"},
)

plot_button = widgets.Button(description="Plot It", button_style="primary", layout=widgets.Layout(width="120px"))
example_button = widgets.Button(description="Load Example", button_style="info", layout=widgets.Layout(width="150px"))
output_area = widgets.Output()

# Accumulate student questions across clicks
student_questions = []  # list of {"question": str, "coords": array, "nearest_idx": int, "sim": float}

MARKER_COLORS = ["#FF0000", "#FF6600", "#9900CC", "#00CC66", "#0066FF", "#CC0066"]

examples = [
    "What is the dress code at Northbrook?",
    "How much does health insurance cost?",
    "Who approved the cloud migration?",
    "What are the Q4 revenue projections?",
]
example_idx = [0]


def on_plot(b):
    q = question_input.value.strip()
    if not q:
        return

    vec = get_embedding(q)
    q_3d = pca.transform(np.array([vec]))

    # Find closest chunk
    sims = [cosine_similarity(vec, chunk_embeddings[j]) for j in range(len(chunk_embeddings))]
    best_idx = int(np.argmax(sims))
    best_sim = sims[best_idx]

    student_questions.append({
        "question": q,
        "coords": q_3d[0],
        "nearest_idx": best_idx,
        "sim": best_sim,
    })

    with output_area:
        clear_output(wait=True)

        # Rebuild figure with all accumulated questions
        fig = build_full_figure()

        for i, sq in enumerate(student_questions):
            color = MARKER_COLORS[i % len(MARKER_COLORS)]
            label = f"Q: {sq['question'][:30]}..."

            fig.add_trace(go.Scatter3d(
                x=[sq["coords"][0]], y=[sq["coords"][1]], z=[sq["coords"][2]],
                mode="markers+text",
                marker=dict(size=12, color=color, symbol="diamond", opacity=1.0, line=dict(width=2, color="black")),
                name=label,
                text=[f"S{i+1}"],
                textposition="top center",
                textfont=dict(size=10, color=color),
                hovertext=[f"{sq['question']}<br>Nearest: {chunk_sources[sq['nearest_idx']]} ({sq['sim']:.4f})"],
                hoverinfo="text",
                showlegend=True,
            ))

            # Dashed line to nearest chunk
            ni = sq["nearest_idx"]
            fig.add_trace(go.Scatter3d(
                x=[sq["coords"][0], coords_3d[ni, 0]],
                y=[sq["coords"][1], coords_3d[ni, 1]],
                z=[sq["coords"][2], coords_3d[ni, 2]],
                mode="lines",
                line=dict(color=color, width=3, dash="dash"),
                showlegend=False, hoverinfo="skip",
            ))

        fig.update_layout(title="Embedding Space: Your Questions Added")
        fig.show()

        sq = student_questions[-1]
        print(f"\nQuestion: {sq['question']}")
        print(f"Closest chunk: {chunk_sources[sq['nearest_idx']]} (type={chunk_doc_types[sq['nearest_idx']]}, sim={sq['sim']:.4f})")
        print(f"Chunk preview: {chunk_texts[sq['nearest_idx']][:120]}...")


plot_button.on_click(on_plot)


def on_example(b):
    idx = example_idx[0] % len(examples)
    question_input.value = examples[idx]
    example_idx[0] += 1


example_button.on_click(on_example)

display(widgets.VBox([
    question_input,
    widgets.HBox([plot_button, example_button]),
    output_area,
]))

---

## 9. Save PCA Transformer

Save the fitted PCA transformer so the walk-along notebook (`session_3_2.ipynb`) can reuse it. Students will load this pickle in their PCA exploration cell to project their own questions without recomputing the PCA fit.

In [ ]:
save_path = Path("pca_transformer.pkl")

# Save the PCA transformer and corpus coordinates together
# so session_3_2.ipynb has everything it needs
pca_bundle = {
    "pca": pca,
    "chunk_coords_3d": coords_3d,
    "chunk_doc_types": chunk_doc_types,
    "chunk_sources": chunk_sources,
    "chunk_texts": chunk_texts,
    "chunk_embeddings": chunk_embeddings,
    "doc_type_colors": DOC_TYPE_COLORS,
    "doc_type_names": DOC_TYPE_NAMES,
}

with open(save_path, "wb") as f:
    pickle.dump(pca_bundle, f)

file_size_kb = save_path.stat().st_size / 1024
print(f"Saved PCA bundle to {save_path}")
print(f"File size: {file_size_kb:.1f} KB")
print()
print("Contents:")
print("  - pca: fitted PCA transformer (call pca.transform() on new embeddings)")
print("  - chunk_coords_3d: pre-computed 3D coordinates for all corpus chunks")
print("  - chunk_doc_types: doc_type label for each chunk")
print("  - chunk_sources: source filename for each chunk")
print("  - chunk_texts: text content of each chunk")
print("  - chunk_embeddings: raw embedding vectors for each chunk")
print("  - doc_type_colors: color mapping for visualization")
print("  - doc_type_names: display names for doc_types")

---

## 10. Takeaway

**Questions live in a different neighborhood than answers.**

When you embed "What is the vacation policy?", the resulting vector is semantically closer to other questions about vacation than it is to the statement "Employees accrue 20 days of PTO per year." The question has the structure of a question -- interrogative phrasing, seeking information. The answer has the structure of a factual assertion -- declarative phrasing, providing information.

Embedding models encode this structural difference. It is not a bug. It is how semantic similarity works.

This is the fundamental limitation of naive cosine similarity for retrieval:
- **Filtering** (today, Session 3.2) narrows the candidate pool, which increases the chance that the nearest neighbor is actually relevant. But within the filtered pool, the gap remains.
- **HyDE** (AI-3) transforms the query to sound like a hypothetical answer before searching, moving the query embedding closer to the answer neighborhood.
- **Question enrichment** (AI-3) adds "what questions does this chunk answer?" to chunks at ingestion time, moving the chunk embeddings closer to the question neighborhood.

Both AI-3 techniques bridge the gap by moving questions and answers closer together in embedding space. Filters, HyDE, and query enrichment are all ways to work around the same underlying problem you just saw.